<a href="https://colab.research.google.com/github/Akakiikent/Sumwork/blob/main/semantic_search_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
from sklearn.manifold import TSNE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
with open("code_corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

with open("eval_questions.json", encoding="utf-8") as f:
    eval_questions = json.load(f)

with open("categories.json", encoding="utf-8") as f:
    categories_raw = json.load(f)["categories"]

category_color = {c["key"]: c["color"] for c in categories_raw}
category_label = {c["key"]: c["label"] for c in categories_raw}

print(f"Фрагментов кода в корпусе: {len(corpus)}")
print(f"Тестовых вопросов: {len(eval_questions)}")

corpus_df = pd.DataFrame(corpus)
corpus_df["category"].value_counts()

In [ ]:

corpus_texts = [f"{c['function_name']}: {c['description']}" for c in corpus]
corpus_ids = [c["id"] for c in corpus]
corpus_categories = [c["category"] for c in corpus]

queries = [q["query"] for q in eval_questions]
correct_ids = [q["correct_chunk_id"] for q in eval_questions]
query_languages = [q["language"] for q in eval_questions]

print("Пример текста для эмбеддинга:", corpus_texts[0])
print("Пример запроса:", queries[0], "-> правильный ответ:", correct_ids[0])

In [ ]:
MODEL_CONFIGS = {
    "MiniLM-multilingual": {
        "name": "paraphrase-multilingual-MiniLM-L12-v2",
        "query_prefix": "",
        "passage_prefix": "",
    },
    "MPNet-multilingual": {
        "name": "paraphrase-multilingual-mpnet-base-v2",
        "query_prefix": "",
        "passage_prefix": "",
    },
    "E5-multilingual (бонус)": {
        "name": "intfloat/multilingual-e5-base",
        "query_prefix": "query: ",
        "passage_prefix": "passage: ",
    },
}